# PAST2HARM — Results Analysis

This notebook reproduces the key figures and tables from the paper.

**Prerequisites**: Run attack scripts first and place `.jsonl` result files in `results/`.


In [ ]:
import sys
sys.path.insert(0, '..')

import json
from pathlib import Path
from collections import defaultdict

import matplotlib.pyplot as plt
import matplotlib
import numpy as np
import pandas as pd
import seaborn as sns

from past2harm.metrics import (
    compute_asr,
    compute_asr_by_budget,
    compute_severity_stats,
    compute_per_category_asr,
    compute_transferability_heatmap,
    find_peak_vulnerability_depth,
)
from past2harm.utils import load_results

matplotlib.rcParams['figure.dpi'] = 120
sns.set_theme(style='whitegrid')
print('Imports OK')

## Load Results

In [ ]:
results_dir = Path('../results')

all_results = {}
for f in sorted(results_dir.glob('*.jsonl')):
    data = load_results(f)
    if data:
        all_results[f.stem] = data
        print(f'{f.stem}: {len(data)} results')

print(f'\nTotal files loaded: {len(all_results)}')

## Table 5: Full ASR by Model, Strategy, Budget K

In [ ]:
budgets = [2, 4, 6, 8]
rows = []
for key, results in all_results.items():
    asr_by_k = compute_asr_by_budget(results, budgets=budgets)
    model = results[0].get('model', key) if results else key
    strategy = results[0].get('strategy', 'unknown') if results else 'unknown'
    row = {'Model': model, 'Strategy': strategy}
    for k in budgets:
        row[f'K={k}'] = f"{asr_by_k[k]*100:.1f}%"
    rows.append(row)

asr_df = pd.DataFrame(rows)
print('ASR Table (Table 5):')
asr_df

## Figure 3: ASR vs Interaction Budget K

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
budgets = [2, 4, 6, 8]

# Group by model
model_groups = defaultdict(dict)
for key, results in all_results.items():
    if results:
        model = results[0].get('model', key)
        strategy = results[0].get('strategy', 'unknown')
        model_groups[model][strategy] = results

models = list(model_groups.keys())[:3]
strategy_styles = {
    'adaptive_pt': ('o-', 'tab:blue', 'Adaptive Past Tense'),
    'past_tense_only': ('s--', 'tab:orange', 'Only Past Tense'),
    'future_tense': ('^:', 'tab:green', 'Future Tense'),
}

for ax, model in zip(axes, models):
    for strategy, results in model_groups[model].items():
        asrs = [compute_asr(results, max_steps=k) * 100 for k in budgets]
        marker, color, label = strategy_styles.get(strategy, ('o-', 'gray', strategy))
        ax.plot(budgets, asrs, marker, color=color, label=label, linewidth=2, markersize=7)
    ax.set_title(model, fontsize=13)
    ax.set_xlabel('Interaction Budget K', fontsize=11)
    ax.set_ylim(0, 105)
    ax.set_xticks(budgets)
    ax.legend(fontsize=9)

axes[0].set_ylabel('ASR (%)', fontsize=12)
plt.suptitle('Attack Success Rate vs Interaction Budget K', fontsize=14)
plt.tight_layout()
plt.savefig('../plots/figure3_asr_vs_budget.png', bbox_inches='tight')
plt.show()

## Figure 4: Depth-Wise Severity Distribution (Box Plot)

In [ ]:
# Use first available results file
if all_results:
    first_key = next(iter(all_results))
    results = all_results[first_key]

    depth_scores = defaultdict(list)
    for r in results:
        for step in r.get('steps', []):
            d = step['depth']
            if d <= 10:
                depth_scores[d].append(step['severity_jailbreak'])

    depths = sorted(depth_scores.keys())
    data = [depth_scores[d] for d in depths]
    medians = [np.median(v) for v in data]

    fig, ax = plt.subplots(figsize=(10, 5))
    bp = ax.boxplot(data, positions=depths, widths=0.5, patch_artist=True,
                    boxprops=dict(facecolor='lightblue', color='navy'),
                    medianprops=dict(color='navy', linewidth=2),
                    whis=[5, 95])
    ax.plot(depths, medians, 'k--', linewidth=1.5, label='Median')
    ax.set_xlabel('Conversation depth', fontsize=12)
    ax.set_ylabel('Jailbreak severity', fontsize=12)
    ax.set_title('Distribution of Jailbreak Severity Across Conversation Depth', fontsize=13)
    ax.set_ylim(0, 1.05)
    ax.legend(fontsize=10)
    plt.tight_layout()
    plt.savefig('../plots/figure4_severity_boxplot.png', bbox_inches='tight')
    plt.show()

    sev_df = compute_severity_stats(results)
    peak = find_peak_vulnerability_depth(sev_df)
    print(f'\nPeak vulnerability depth: {peak}')
    print(sev_df[['Mean', 'Std', 'Median']].to_string())
else:
    print('No results loaded. Run attacks first.')

## Table 6: Per-Category ASR

In [ ]:
if all_results:
    first_key = next(iter(all_results))
    results = all_results[first_key]
    cat_df = compute_per_category_asr(results, max_steps=8)
    cat_df = cat_df.sort_values('ASR_%', ascending=False)
    print(f'Per-category ASR ({first_key}):')
    cat_df

## Figure 5b: Transferability Heatmap

In [ ]:
# Look for transfer result files (named transfer_<src>_to_<tgt>.jsonl)
transfer_results = {}
for f in results_dir.glob('transfer_*.jsonl'):
    data = load_results(f)
    if data:
        src = data[0].get('src_model', 'unknown')
        tgt = data[0].get('tgt_model', 'unknown')
        transfer_results[(src, tgt)] = data

if transfer_results:
    heatmap_df = compute_transferability_heatmap(transfer_results, max_steps=8)

    fig, ax = plt.subplots(figsize=(6, 5))
    sns.heatmap(
        heatmap_df.astype(float),
        annot=True, fmt='.1f', cmap='YlOrRd',
        vmin=0, vmax=100,
        ax=ax,
        annot_kws={'size': 14, 'weight': 'bold'},
    )
    ax.set_title('Transferability Heatmap — ASR (%)', fontsize=13)
    ax.set_xlabel('Target Model')
    ax.set_ylabel('Source Model')
    plt.tight_layout()
    plt.savefig('../plots/figure5b_transferability.png', bbox_inches='tight')
    plt.show()
    print(heatmap_df.to_string())
else:
    print('No transfer results found. Run transferability.py first.')